In [64]:
import json
import os
import re

from google import genai

In [ ]:
client = genai.Client(api_key="<YOUR GOOGLE_API_KEY>")

In [66]:
domain_agnostic_profiles = [
    "ComputationalWorkflow", 
    "FormalParameter", 
    "DataCatalog",
    "Dataset", 
    "Course", 
    "CourseInstance", 
]

In [67]:
for domain_agnostic_profile in domain_agnostic_profiles:
    
    with open("data/prompts/prompt.txt", "r") as file:
        prompt = file.read().replace("DOMAIN_AGNOSTIC_PROFILE", domain_agnostic_profile)

    response = client.models.generate_content(
        model="gemini-2.5-flash", contents=prompt
    )
    
    with open("../docs/guidance/" + domain_agnostic_profile + ".md", "w") as f:
        f.write(response.text)

In [68]:
def convert_markdown_to_json(
    input_path, 
    output_path
):
    with open(input_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # Extract the title
    title_match = re.search(r'# Guidance for using (.*?)\n', content)
    title = title_match.group(1).strip() if title_match else "Title Not Found"

    # Extract the hierarchy
    hierarchy_match = re.search(r'^(\[.+?\]\(.+?\)\s*>.+)', content, re.MULTILINE)
    hierarchy = hierarchy_match.group(1).strip() if hierarchy_match else "Hierarchy Not Found"

    # Extract and clean the JSON block
    json_block_match = re.search(r'```json\s*\n(.*?)\n```', content, re.DOTALL)
    if not json_block_match:
        raise ValueError("No JSON code block found.")

    json_string = json_block_match.group(1)
    
    # Remove markdown links from the JSON string
    json_string = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', json_string)
    
    json_data = json.loads(json_string)

    output_data = {
        "title": title,
        "hierarchy": hierarchy,
        "json": json_data
    }

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, indent=2)


In [69]:
for domain_agnostic_profile in domain_agnostic_profiles:
    input_path = f"../docs/guidance/{domain_agnostic_profile}.md"
    output_path = f"data/output_options/{domain_agnostic_profile}/option.json"

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    convert_markdown_to_json(input_path, output_path)